In [ ]:
import os, sys
import numpy as np, pandas as pd

# Environment bootstrap: Colab (mount Drive) or a local checkout.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    os.chdir("/content/drive/MyDrive/volatility-forecast")
except ModuleNotFoundError:
    p = os.path.abspath(os.getcwd())          # find the repo root locally
    while not os.path.isdir(os.path.join(p, "src")):
        parent = os.path.dirname(p)
        if parent == p:
            raise FileNotFoundError("repo root with src/ not found")
        p = parent
    os.chdir(p)

for m in list(sys.modules):
    if m.startswith("src."):
        del sys.modules[m]
from src.splits import walk_forward_splits
from src.drift import ks_drift, psi

# Load raw dataset, then build the SAME work frame the 03/04 models trained on.
df = pd.read_csv("data/processed/dataset.csv", index_col=0, parse_dates=True)

feat_all = ['qqq_ret','hyg_ret','lqd_ret','tlt_ret','gld_ret','vix_lvl','vix_chg',
            'tnx_lvl','tnx_chg','irx_lvl','irx_chg','slope_lvl','slope_chg',
            'credit_lvl','credit_chg','rv1','rv5','rv21']
tgt = ['y_rv1','y_rv5','y_rv21']
work = df.dropna(subset=feat_all + tgt)          # 3831-row shared frame (head21+tail21 trimmed)
assert len(work) == 3831, f"expected 3831, got {len(work)}"   # guard: must match model frame

FEATURES = [
    "rv1", "rv5", "rv21",          # realized vol: mean-reverting, level OK
    "vix_lvl",                     # implied vol: mean-reverting, level OK
    "tnx_chg",                     # 10y yield CHANGE (level trended -> saturates)
    "credit_chg",                  # credit spread CHANGE (level trended)
    "slope_chg",                   # curve slope CHANGE (level trended)
    "tlt_ret", "gld_ret",          # safe-haven returns (stationary)
    "hyg_ret", "lqd_ret",          # credit ETF returns (stationary)
]
splits = list(walk_forward_splits(len(work)))    # 11-fold over work frame
print(f"n_folds={len(splits)}, n_features={len(FEATURES)}, n={len(work)}")
print("missing:", [c for c in FEATURES if c not in work.columns])

# date map sanity -- must show f05=COVID(2020-03), f07=2022bear(2022-03)
for k,(tr,te) in enumerate(splits):              # 0-indexed: f00..f10
    print(f"f{k:02d}: test {work.index[te[0]].date()} -> {work.index[te[-1]].date()}")

In [2]:
TRAIL_N = 252
rows = []
for k, (tr_idx, te_idx) in enumerate(splits):    # 0-indexed f00..f10
    ref_windows = {"expanding": tr_idx, "trailing": tr_idx[-TRAIL_N:]}
    for feat in FEATURES:
        test_x = work[feat].values[te_idx]       # work, not df
        for ref_type, ref_idx in ref_windows.items():
            ref_x = work[feat].values[ref_idx]   # work, not df
            D, p = ks_drift(ref_x, test_x)
            rows.append({"fold": k, "feature": feat, "ref": ref_type,
                         "ks_D": D, "ks_p": p, "psi": psi(ref_x, test_x)})
res = pd.DataFrame(rows)
pivot = res[res.ref == "trailing"].pivot(index="feature", columns="fold", values="psi")
print(pivot.round(3))

fold           0      1      2      3      4      5      6       7       8   \
feature                                                                       
credit_chg  0.167  0.092  0.290  0.068  0.091  0.209  0.381   0.327   0.199   
gld_ret     0.042  0.118  0.300  0.187  0.140  0.139  0.178   0.096   0.130   
hyg_ret     0.257  0.087  0.810  0.174  0.105  0.409  0.792   1.315   0.533   
lqd_ret     0.116  0.092  0.081  0.024  0.168  0.176  0.049   0.595   0.176   
rv1         0.109  0.128  0.175  0.626  0.217  0.471  0.379   0.565   1.660   
rv21        2.790  1.732  2.945  5.867  3.726  3.688  2.958  10.842  10.094   
rv5         0.312  0.570  0.520  1.754  1.420  1.140  1.608   5.785   7.466   
slope_chg   0.166  0.097  0.100  0.041  0.057  0.074  0.143   0.460   0.135   
tlt_ret     0.239  0.204  0.131  0.047  0.192  0.119  0.029   0.354   0.062   
tnx_chg     0.133  0.138  0.134  0.039  0.120  0.038  0.065   0.740   0.074   
vix_lvl     1.597  0.320  3.963  7.890  0.989  7.710

In [3]:
# expanding vs trailing: where do the two ref windows disagree?
# Focus on the regime signals (vix_lvl, rv21) at key folds.
cmp = (res[res.feature.isin(["vix_lvl", "rv21", "rv5"])]
       .pivot_table(index=["feature", "fold"], columns="ref", values="psi")
       .round(3))
print(cmp)

ref           expanding  trailing
feature fold                     
rv21    0         1.373     2.790
        1         0.550     1.732
        2         3.055     2.945
        3         1.911     5.867
        4         0.284     3.726
        5         3.671     3.688
        6         0.557     2.958
        7         9.618    10.842
        8         3.188    10.094
        9         2.837     0.370
        10        0.121     0.468
rv5     0         0.076     0.312
        1         0.217     0.570
        2         0.732     0.520
        3         0.524     1.754
        4         0.092     1.420
        5         1.116     1.140
        6         0.106     1.608
        7         5.994     5.785
        8         0.786     7.466
        9         0.307     0.203
        10        0.032     0.284
vix_lvl 0         0.109     1.597
        1         0.375     0.320
        2         5.226     3.963
        3         0.338     7.890
        4         1.623     0.989
        5     

In [ ]:
# 05 output: per-fold × feature × ref-type drift table -> stage CSV for 06
out_path = "data/processed/covariate_shift.csv"
res.to_csv(out_path, index=False)
print(f"saved {len(res)} rows -> {out_path}")
print(res.groupby("ref")["psi"].describe().round(3))   # sanity: trailing wider spread

In [5]:
# --- 05 binarization: per-feature relative threshold + fold-level aggregation ---
# CALM defined by 03/04 ground truth (model beats persistence), NOT by P(X) -> no circularity.
CALM_FOLDS = [0, 1, 2, 3, 4, 6, 8, 9, 10]   # f05=COVID, f07=2022bear excluded
THR_PSI = 0.25      # absolute PSI floor (standard convention)
THR_MULT = 3.0      # must exceed 3x its own calm baseline
THR_KSP = 0.01      # KS p-value: reject "same dist" at 1%

tr = res[res.ref == "trailing"].copy()

# per-feature calm baseline = median PSI over calm folds (feature-specific scale)
calm_base = (tr[tr.fold.isin(CALM_FOLDS)]
             .groupby("feature")["psi"].median().rename("calm_psi"))
tr = tr.merge(calm_base, on="feature")

# fire if (absolute PSI large AND >=3x own baseline) OR (KS rejects at 1%)
tr["psi_fire"] = (tr.psi > THR_PSI) & (tr.psi > THR_MULT * tr.calm_psi)
tr["ks_fire"]  = tr.ks_p < THR_KSP
tr["fire"]     = tr.psi_fire | tr.ks_fire

# fold-level: how many features fired
fold_fire = (tr.groupby("fold")
             .agg(n_fire=("fire", "sum"),
                  fired=("feature", lambda s: list(s[tr.loc[s.index, "fire"]])))
             .reset_index())
print(fold_fire.to_string(index=False))

COV_SHIFT_SET = set(fold_fire.loc[fold_fire.n_fire >= 2, "fold"])
print("\ncovariate-shift POSITIVE set (>=2 feats):", sorted(COV_SHIFT_SET))
print("06 concept-drift TRUE positive (ground truth): {7}")
print("f05 in cov-shift set?", 5 in COV_SHIFT_SET, "(should be True = false-alarm proof)")
print("f07 in cov-shift set?", 7 in COV_SHIFT_SET, "(should be True)")

 fold  n_fire                                                                    fired
    0       4                                            [rv5, rv21, vix_lvl, hyg_ret]
    1       3                                                     [rv5, rv21, vix_lvl]
    2       5                                       [rv1, rv5, rv21, vix_lvl, hyg_ret]
    3       4                                                [rv1, rv5, rv21, vix_lvl]
    4       5                                       [rv1, rv5, rv21, vix_lvl, lqd_ret]
    5       5                                       [rv1, rv5, rv21, vix_lvl, hyg_ret]
    6       5                                       [rv1, rv5, rv21, vix_lvl, hyg_ret]
    7       9 [rv1, rv5, rv21, vix_lvl, tnx_chg, slope_chg, tlt_ret, hyg_ret, lqd_ret]
    8       5                                       [rv1, rv5, rv21, vix_lvl, hyg_ret]
    9       4                                        [rv21, vix_lvl, hyg_ret, lqd_ret]
   10       3                              

In [6]:
# KS p-value is meaningless at n=252 (rejects everything). Drop ks_fire from union.
# Use PSI relative threshold alone; keep KS D only as a reported effect-size column.
tr["fire"] = tr["psi_fire"]          # PSI-only: absolute>0.25 AND >=3x own calm baseline

fold_fire = (tr.groupby("fold")
             .agg(n_fire=("fire", "sum"),
                  fired=("feature", lambda s: list(s[tr.loc[s.index, "fire"]])))
             .reset_index())
print(fold_fire.to_string(index=False))

COV_SHIFT_SET = set(fold_fire.loc[fold_fire.n_fire >= 2, "fold"])
print("\ncov-shift POSITIVE set (>=2 feats, PSI-only):", sorted(COV_SHIFT_SET))
print("f05 in set?", 5 in COV_SHIFT_SET, "| f07 in set?", 7 in COV_SHIFT_SET)
print("calm folds also firing?:", sorted(COV_SHIFT_SET - {5, 7}))

 fold  n_fire                                                  fired
    0       0                                                     []
    1       0                                                     []
    2       1                                              [hyg_ret]
    3       2                                             [rv1, rv5]
    4       0                                                     []
    5       0                                                     []
    6       1                                              [hyg_ret]
    7       7 [rv1, rv5, rv21, tnx_chg, slope_chg, hyg_ret, lqd_ret]
    8       3                                       [rv1, rv5, rv21]
    9       1                                              [lqd_ret]
   10       0                                                     []

cov-shift POSITIVE set (>=2 feats, PSI-only): [3, 7, 8]
f05 in set? False | f07 in set? True
calm folds also firing?: [3, 8]


In [7]:
# Per-fold covariate-shift INTENSITY: aggregate PSI across features (regime signals).
# Thesis is about RANKING (can P(X) separate f05 from f07?), not set membership.
tr = res[res.ref == "trailing"].copy()

# normalize each feature's PSI to its own calm-fold scale, then aggregate per fold
calm_base = (tr[tr.fold.isin(CALM_FOLDS)]
             .groupby("feature")["psi"].median().rename("calm_psi"))
tr = tr.merge(calm_base, on="feature")
tr["psi_norm"] = tr["psi"] / tr["calm_psi"]      # ratio vs own calm baseline

# two aggregate scores per fold
score = (tr.groupby("fold")
         .agg(psi_sum=("psi", "sum"),            # raw total shift
              psi_norm_mean=("psi_norm", "mean"), # avg multiple-of-calm
              vix_psi=("psi", lambda s: s[tr.loc[s.index,"feature"]=="vix_lvl"].iloc[0]),
              rv21_psi=("psi", lambda s: s[tr.loc[s.index,"feature"]=="rv21"].iloc[0]))
         .reset_index()
         .sort_values("psi_norm_mean", ascending=False))
print(score.round(2).to_string(index=False))
print("\nf05 (COVID) and f07 (2022bear) ranks among 11 folds:")
ranked = score.reset_index(drop=True)
print("  f05 rank:", ranked.index[ranked.fold==5][0]+1, "/ 11")
print("  f07 rank:", ranked.index[ranked.fold==7][0]+1, "/ 11")

 fold  psi_sum  psi_norm_mean  vix_psi  rv21_psi
    7    25.90           4.13     4.82     10.84
    8    26.85           3.31     6.32     10.09
    6    10.85           1.52     4.27      2.96
    5    14.17           1.46     7.71      3.69
    2     9.45           1.38     3.96      2.95
    3    16.72           1.35     7.89      5.87
    9     2.78           1.22     0.31      0.37
    4     7.23           1.10     0.99      3.73
    0     5.93           0.96     1.60      2.79
    1     3.58           0.79     0.32      1.73
   10     4.87           0.71     3.22      0.47

f05 (COVID) and f07 (2022bear) ranks among 11 folds:
  f05 rank: 4 / 11
  f07 rank: 1 / 11
